# Chapter 15: Evaluating Agents Properly
## Topic 2: Task-Level vs Step-Level Metrics

**One notebook. Theory + Code together.**


## Part A: Theory

### 1. Concept, Intuition, and Why This Exists

- Topic 1 established that agent evaluation needs to look beyond the final answer to the process that produced it. This topic gives that principle its actual, precise measurement vocabulary: **task-level metrics** evaluate whether the agent achieved the overall goal (conceptually similar to a classifier's single output-correctness check, just applied to an agent's end result), while **step-level metrics** evaluate whether each individual decision within the process — each tool call, each intermediate judgment — was itself correct, independent of whether the final task outcome happened to be correct.
- This distinction is precisely what Module 1's demonstration in Topic 1 made concrete without yet naming it formally: "final-answer-only evaluation" in that topic's code *is* task-level evaluation, and "process-aware evaluation" *is* a specific instance of step-level evaluation (checking one specific step — the tool call's actual result — rather than only the task's final outcome).
- The core, generalizable insight this topic adds beyond Topic 1's single demonstration: task-level and step-level metrics are not competing alternatives where one is simply "better" — they answer genuinely different questions and should be reported together, exactly the same "report metrics separately, never collapse into one blended score" discipline this notebook series has established repeatedly (Chapter 9 Topic 7's hallucination-rate segmentation, Chapter 14 Topic 3's four separate RAGAS metrics) — now applied specifically to the task-vs-step distinction for agent evaluation.


### 2. Internal Working — Step by Step

**Defining and computing task-level and step-level metrics precisely:**

1. **Task-level accuracy** asks: did the agent's overall final output match the expected final outcome for this test case? This is computed exactly like Chapter 1's classifier accuracy — one comparison per test case, aggregated into a percentage — just applied to an agent's end-to-end final answer rather than a classifier's single-step output.
2. **Step-level accuracy**, by contrast, requires decomposing a single test case's evaluation into multiple, separately-checkable sub-decisions: was the correct tool selected (Chapter 10 Topic 6's tool-selection concern), was it called with correct arguments (Chapter 10 Topic 4's schema-quality concern), was its result correctly interpreted, and so on — each of these is its own pass/fail check, and step-level accuracy aggregates across these individual, finer-grained checks rather than across whole test cases.
3. **The two metrics can diverge in genuinely informative ways**, and reading that divergence correctly is this topic's real, practical payoff: high task-level accuracy with lower step-level accuracy suggests the agent is reaching correct final outcomes despite some unreliable intermediate steps (exactly Topic 1's "right answer, wrong process, got lucky" pattern) — a warning sign the current evaluation set's specific cases haven't yet exposed the risk this instability represents. Conversely, high step-level accuracy with lower task-level accuracy suggests each individual step is being executed correctly, but something in how those correct steps combine, or in the final generation step itself, is producing incorrect overall outcomes.
4. **Step-level metrics require finer-grained ground truth than task-level metrics do** — directly connecting to Topic 1's own observation about the added cost of rigorous agent evaluation-set construction: a step-level check needs to know not just the correct final answer, but the correct (or acceptable) tool calls, arguments, and intermediate results for each test case, a genuinely richer evaluation-set requirement.


### 3. How This Is Implemented in This Project

- Topic 1's own Module 3 code is directly reusable as an illustration of this exact distinction: its "final-answer-only" evaluation function computes task-level accuracy, and its "process-aware" evaluation function computes a specific instance of step-level accuracy (checking the tool call's actual result correctness) — this topic's contribution is generalizing that single demonstration into the full task-level/step-level framework and showing how to decompose step-level checks further, across multiple distinct steps rather than just one.
- Chapter 10 Topic 7's own three-category test suite (triggering/selection, argument-correctness, result-handling) is, in retrospect, already a step-level evaluation framework — this topic names that structure explicitly and shows how it complements, rather than replaces, a task-level (final-answer) accuracy measurement that should be tracked alongside it, not instead of it.
- This project's agent, evaluated properly, should report both a task-level accuracy figure (did the classification/answer come out right, end to end) and a decomposed step-level accuracy breakdown (was retrieval triggered correctly, was the right tool selected, were arguments correct, was memory correctly retrieved and used) — exactly the multi-dimensional reporting this topic's theory argues for, rather than either metric alone.


### 4. Real-World Issues, Edge Cases, Debugging, Monitoring, Scaling, Latency, Cost, Security, Deployment

- **Reporting only task-level accuracy hides exactly the risk Topic 1's demonstration made concrete** — a high task-level accuracy figure provides no visibility into whether that accuracy is being achieved through a reliable process or through Topic 1's "right answer, wrong process, got lucky" pattern, which could fail unpredictably on slightly different real queries.
- **Reporting only step-level accuracy, without task-level accuracy alongside it, misses whether correctly-executed individual steps actually combine into correct overall outcomes** — a step-level-only view could show every individual tool call behaving correctly while still missing a systematic error in how the final generation step synthesizes those correct results, since that synthesis step itself is a place errors can be introduced independent of any single tool call's correctness.
- **Divergence between the two metrics is diagnostically valuable and needs specific investigation, not just noting** — exactly Chapter 9 Topic 7's own "don't just report the aggregate, investigate the specific pattern" discipline, now applied to interpreting a task-level/step-level gap rather than a segment-level gap in a single metric.
- **Debugging requires knowing which specific step-level metric is driving an overall step-level accuracy figure down** — reporting one blended step-level number (across tool selection, argument correctness, and result interpretation combined) reproduces exactly the same "collapsed aggregate hides the actual failure location" mistake this notebook series has repeatedly warned against, now within step-level metrics themselves, not just between task-level and step-level.
- **Monitoring in production:** tracking both task-level accuracy and a decomposed step-level breakdown over time (not just once, during initial evaluation) reveals whether a pipeline change (a new tool, a new prompt) improves or regresses each dimension independently — directly connecting to Topic 5's regression-tracking discipline.


### 5. Design Decisions, Trade-offs, and Real-Time Dilemmas

- **How finely to decompose step-level metrics:** a coarser decomposition (one blended "process correctness" check) is cheaper to construct but provides less diagnostic specificity; a finer decomposition (separate checks for tool selection, argument correctness, result interpretation) provides much more actionable debugging information at the cost of a richer, more expensive evaluation-set construction effort — the right granularity should be informed by which specific step-level failures have actually been observed or are suspected, not decomposed maximally by default regardless of whether that granularity is actually useful.
- **Weighting task-level vs step-level metrics when they disagree about overall system quality:** neither should simply override the other — a genuine disagreement (high task-level, low step-level, or vice versa) is itself diagnostically important information, and the right response is investigating the specific disagreement's cause, not picking whichever metric looks more favorable to report.
- **How much step-level evaluation infrastructure to build immediately vs incrementally:** mirroring Topic 1's own incremental-construction principle, building step-level checks specifically for steps where a real, observed or suspected failure exists (rather than attempting comprehensive step-level coverage across every conceivable intermediate decision from day one) is a more pragmatic, cost-effective approach.


### 6. Alternatives and When to Use Each

- **Task-level accuracy alone:** appropriate only very early in a project, or for a genuinely simple, single-step system where no meaningful intermediate process exists to evaluate separately — insufficient for any real agentic system, per Topic 1's core argument.
- **Step-level accuracy alone, without task-level accuracy:** risks missing failures in how correctly-executed individual steps combine into final outcomes — not recommended as a standalone evaluation approach.
- **Both task-level and step-level metrics reported together (this topic's recommended default):** the right approach for any genuinely agentic system, providing both an overall outcome measure and specific, actionable process-level diagnostic detail.


### 7. Common Mistakes and Production Failures

- Reporting only task-level (final-answer) accuracy, missing the process-reliability risk Topic 1's demonstration made concrete — a system that looks reliable on the current evaluation set but is actually succeeding through an unreliable process.
- Reporting only step-level accuracy without task-level accuracy alongside it, missing failures in how correctly-executed steps combine into the final outcome.
- Collapsing multiple distinct step-level checks (tool selection, argument correctness, result interpretation) into one blended step-level number, losing the specific diagnostic value finer-grained decomposition provides.
- Not investigating a genuine divergence between task-level and step-level metrics, treating it as noise rather than the diagnostically valuable signal it actually represents.
- Attempting comprehensive, maximally fine-grained step-level evaluation coverage from day one, rather than building it incrementally around actually-observed or suspected failure modes.


### 8. Lead-Level Interview Questions

**Basic**

- Q: What's the difference between task-level and step-level metrics for agent evaluation?
  A: Task-level metrics evaluate whether the agent's overall final output matched the expected outcome — similar to classifier accuracy, applied to an agent's end result. Step-level metrics evaluate whether each individual decision within the process (tool selection, argument correctness, result interpretation) was itself correct, independent of whether the final task outcome happened to be correct.

- Q: Why should task-level and step-level metrics be reported together, rather than choosing one over the other?
  A: They answer genuinely different questions and can diverge in diagnostically informative ways. High task-level accuracy with lower step-level accuracy suggests the agent is reaching correct outcomes despite unreliable intermediate steps — a real risk that could fail unpredictably on different queries. Reporting only one metric hides exactly the information the other would reveal.

**Intermediate**

- Q: How does Chapter 10 Topic 7's three-category test suite relate to this topic's step-level metrics concept?
  A: That test suite (triggering/selection, argument-correctness, result-handling) is, in retrospect, already a step-level evaluation framework, checking specific intermediate decisions rather than only final outcomes — this topic names that underlying structure explicitly and clarifies that it should be reported alongside, not instead of, task-level accuracy.

- Q: What does it mean when step-level accuracy is high but task-level accuracy is lower for the same evaluation set?
  A: This suggests each individual step (tool selection, correct arguments, correct intermediate results) is being executed correctly, but something in how those correct steps combine — or in the final generation step itself — is producing incorrect overall outcomes. This points investigation toward the synthesis/generation layer rather than toward any individual tool or retrieval step.

**Advanced**

- Q: Design a step-level metric decomposition for this project's actual agent, specifying what each sub-metric would check.
  A: Tool-selection accuracy (did the agent choose to call the correct tool, or correctly choose not to call any tool, per Chapter 10 Topic 6's multi-tool selection concern), argument-correctness accuracy (were the tool's arguments well-formed and correct, per Chapter 10 Topic 4's schema-quality concern), result-interpretation accuracy (did the agent correctly incorporate the tool's actual result into its reasoning, catching exactly the kind of flaw Topic 1's demonstration exposed), and retrieval-triggering accuracy (did the agent correctly decide whether retrieval was needed at all, per Chapter 13 Topic 2's confidence-based triggering). Each should be measured and reported as its own distinct figure, not blended into one step-level number.

- Q: A teammate proposes using only task-level accuracy for a quarterly quality report, arguing step-level detail is too granular for executive stakeholders. How do you respond?
  A: Task-level accuracy is a reasonable headline number for a broad audience, but it should never be the *only* metric tracked internally, since it structurally cannot reveal the process-reliability risk Topic 1 demonstrated concretely — a system could show consistently high task-level accuracy while harboring a serious, unaddressed process flaw that eventually surfaces unpredictably. Step-level metrics can be summarized at an appropriate level of detail for stakeholder reporting (perhaps as a single "process reliability" indicator) without being discarded from the team's own internal tracking and debugging practice.

**Scenario-based**

- Q: Your agent's task-level accuracy has remained stable at 92% for several weeks, but step-level metrics show tool-selection accuracy declining from 95% to 85% over the same period. How do you interpret this and what would you do?
  A: This is precisely the divergence pattern this topic identifies as diagnostically important — task-level accuracy staying stable while step-level tool-selection accuracy declines suggests the agent may increasingly be reaching correct final answers despite an increasingly unreliable tool-selection process, exactly Topic 1's "right answer, wrong process" risk, now potentially worsening over time. This warrants immediate investigation into why tool-selection accuracy is declining (a schema change, a prompt regression, a shift in real query patterns) before this instability eventually produces a task-level accuracy drop too, rather than waiting for the stable task-level number to eventually reveal the problem on its own.

**Follow-up questions to expect:**

- "How would you decide which step-level metrics are worth the cost of building evaluation infrastructure for?"
- "What would you do if two different step-level metrics moved in opposite directions after a pipeline change?"


### 9. Hidden Concepts / Prerequisites Worth Knowing

- **The task-level/step-level distinction is a specific instance of a general "outcome metrics vs process metrics" pattern** appearing broadly in performance measurement and quality assurance across many fields, not unique to agent evaluation — recognizing this connects the specific technique to a much more general measurement-design principle.
- **Divergence between two related metrics being diagnostically valuable, rather than something to resolve by picking one, is a recurring theme across this entire notebook series** — Chapter 9 Topic 7's faithfulness/relevance/correctness framework and Chapter 14 Topic 3's four separate RAGAS metrics both already demonstrated this same principle in different contexts; this topic is another instance of it, applied specifically to agent evaluation's task-vs-step distinction.
- **This topic directly sets up Topic 3's specific focus**: "did it call the right tool" is precisely one particular, especially important step-level metric this topic's framework makes room for — Topic 3 goes deep on that one specific, high-value step-level check.

### 10. Quick Revision Summary (for last-minute interview prep)

> Task-level metrics evaluate whether an agent's overall final output matched the expected outcome, computed similarly to classifier accuracy but applied to an agent's end-to-end result. Step-level metrics evaluate whether each individual decision within the agent's process — tool selection, argument correctness, result interpretation — was itself correct, independent of the final outcome. These aren't competing alternatives; they answer genuinely different questions and should be reported together, following this notebook series' repeated "never collapse distinct signals into one number" discipline. Divergence between them is diagnostically valuable: high task-level accuracy with lower step-level accuracy reveals exactly the "right answer, wrong process, got lucky" risk Topic 1 demonstrated concretely, while high step-level accuracy with lower task-level accuracy points toward a problem in how correctly-executed steps combine or in the final generation step itself. Chapter 10 Topic 7's three-category test suite was already, in retrospect, a step-level evaluation framework — this topic names that structure explicitly and establishes that it should complement, not replace, task-level accuracy tracking, setting up Topic 3's deeper focus on one particularly important step-level check: whether the agent called the right tool at all.


### Module 1: A Test Set Where Task-Level and Step-Level Metrics Genuinely Diverge

Build test cases covering both divergence patterns: high task-level/low step-level (lucky correct answers), and high step-level/low task-level (correct steps, wrong synthesis).

In [1]:

# ------------------------------------------------------------------
# MODULE 1: Test cases producing GENUINE task/step divergence
# ------------------------------------------------------------------

FD_RECORDS = {
    "BJ2019FD7717": {"customer_name": "Shobha Chopra", "amount": 391000},
    "BJ2022FD6918": {"customer_name": "Anita Mishra", "amount": 160000},
}

def lookup_flawed(reference_number: str) -> dict:
    """Same FLAWED fuzzy-fallback tool from Topic 1."""
    if reference_number in FD_RECORDS:
        return {"found": True, **FD_RECORDS[reference_number]}
    for ref, record in FD_RECORDS.items():
        if ref[:10] == reference_number[:10]:
            return {"found": True, **record}
    return {"found": False}


def agent_with_synthesis_bug(reference_number: str, correct_tool_result: dict) -> str:
    """Simulates an agent whose TOOL CALL and TOOL RESULT are both
    genuinely correct, but whose FINAL SYNTHESIS step has a real bug
    -- it reports the WRONG amount (an off-by-one-field mistake),
    demonstrating the 'correct steps, wrong final answer' divergence
    direction."""
    if not correct_tool_result["found"]:
        return "We could not find that FD reference number."
    # BUG: synthesis step accidentally reports a HARDCODED wrong amount
    return f"Your FD amount is Rs 999,999, registered to {correct_tool_result['customer_name']}."


# TEST SET A: designed to show HIGH task-level, LOW step-level
# (the FLAWED tool's fuzzy-fallback bug produces correct-looking
# final answers for genuinely-existing references)
TEST_SET_TASK_HIGH_STEP_LOW = [
    {"reference": "BJ2019FD7717", "expected_amount": 391000, "expected_found": True},
    {"reference": "BJ2022FD6918", "expected_amount": 160000, "expected_found": True},
]

# TEST SET B: designed to show HIGH step-level, LOW task-level
# (tool call and result are CORRECT, but synthesis is buggy)
TEST_SET_STEP_HIGH_TASK_LOW = [
    {"reference": "BJ2019FD7717", "expected_amount": 391000, "expected_found": True},
    {"reference": "BJ2022FD6918", "expected_amount": 160000, "expected_found": True},
]

print("=" * 70)
print("MODULE 1: TEST SETS DESIGNED FOR GENUINE METRIC DIVERGENCE")
print("=" * 70)
print(f"\nTest Set A ({len(TEST_SET_TASK_HIGH_STEP_LOW)} cases): tests the FLAWED tool")
print(f"  -- expected pattern: HIGH task-level, LOW step-level accuracy")
print(f"\nTest Set B ({len(TEST_SET_STEP_HIGH_TASK_LOW)} cases): tests a synthesis bug")
print(f"  -- expected pattern: HIGH step-level, LOW task-level accuracy")

print("\nModule 1 complete. Run Module 2 next.")


MODULE 1: TEST SETS DESIGNED FOR GENUINE METRIC DIVERGENCE

Test Set A (2 cases): tests the FLAWED tool
  -- expected pattern: HIGH task-level, LOW step-level accuracy

Test Set B (2 cases): tests a synthesis bug
  -- expected pattern: HIGH step-level, LOW task-level accuracy

Module 1 complete. Run Module 2 next.


### Module 2: Computing Task-Level and Step-Level Accuracy Separately

Implement both metrics as real, distinct computations, and run them against Test Set A to show the first divergence pattern.

In [2]:

# ------------------------------------------------------------------
# MODULE 2: Task-level vs step-level accuracy, computed separately
# ------------------------------------------------------------------

def compute_task_level_accuracy(test_cases: list, agent_final_answer_correct_fn) -> float:
    """Checks ONLY whether the final answer matches expectations."""
    correct = sum(1 for case in test_cases if agent_final_answer_correct_fn(case))
    return correct / len(test_cases)


def compute_step_level_accuracy(test_cases: list, tool_result_correct_fn) -> float:
    """Checks whether the underlying TOOL CALL's actual result was
    correct -- independent of what the final synthesized answer said."""
    correct = sum(1 for case in test_cases if tool_result_correct_fn(case))
    return correct / len(test_cases)


# --- evaluating the FLAWED tool against Test Set A ---
def task_check_flawed_tool(case: dict) -> bool:
    result = lookup_flawed(case["reference"])
    if not result["found"]:
        return not case["expected_found"]
    # naive final-answer check: does the agent's constructed sentence
    # simply CONTAIN a plausible amount? (a weak, task-level-only check)
    return result["amount"] == case["expected_amount"] or True  # simulates a "looks fine" pass

def step_check_flawed_tool(case: dict) -> bool:
    result = lookup_flawed(case["reference"])
    return result["found"] == case["expected_found"] and (
        not case["expected_found"] or result["amount"] == case["expected_amount"]
    )


task_accuracy_A = compute_task_level_accuracy(TEST_SET_TASK_HIGH_STEP_LOW, task_check_flawed_tool)
step_accuracy_A = compute_step_level_accuracy(TEST_SET_TASK_HIGH_STEP_LOW, step_check_flawed_tool)

print("=" * 70)
print("TEST SET A RESULTS -- FLAWED TOOL")
print("=" * 70)
print(f"Task-level accuracy: {task_accuracy_A:.2f}")
print(f"Step-level accuracy: {step_accuracy_A:.2f}")

print("\nNOTE: this test set only includes genuinely-existing references")
print("(the flaw only surfaces on a TYPO, per Topic 1) -- so BOTH metrics")
print("look identical here. Let's add the typo case explicitly to see the")
print("REAL divergence Topic 1 demonstrated.")

TEST_SET_A_WITH_TYPO = TEST_SET_TASK_HIGH_STEP_LOW + [
    {"reference": "BJ2019FD771X", "expected_amount": None, "expected_found": False},
]

task_accuracy_A2 = compute_task_level_accuracy(TEST_SET_A_WITH_TYPO, task_check_flawed_tool)
step_accuracy_A2 = compute_step_level_accuracy(TEST_SET_A_WITH_TYPO, step_check_flawed_tool)

print(f"\nWITH the typo case added ({len(TEST_SET_A_WITH_TYPO)} total cases):")
print(f"  Task-level accuracy: {task_accuracy_A2:.2f} (the naive check still 'passes' since SOME answer came out)")
print(f"  Step-level accuracy: {step_accuracy_A2:.2f} (correctly catches the flaw on the typo case)")

print("\nModule 2 complete. Run Module 3 next.")


TEST SET A RESULTS -- FLAWED TOOL
Task-level accuracy: 1.00
Step-level accuracy: 1.00

NOTE: this test set only includes genuinely-existing references
(the flaw only surfaces on a TYPO, per Topic 1) -- so BOTH metrics
look identical here. Let's add the typo case explicitly to see the
REAL divergence Topic 1 demonstrated.

WITH the typo case added (3 total cases):
  Task-level accuracy: 1.00 (the naive check still 'passes' since SOME answer came out)
  Step-level accuracy: 0.67 (correctly catches the flaw on the typo case)

Module 2 complete. Run Module 3 next.


### Module 3: The Opposite Divergence — Correct Steps, Wrong Final Synthesis

Run Test Set B against the synthesis-bug agent, showing the OPPOSITE divergence pattern: step-level accuracy stays high while task-level accuracy drops.

In [3]:

# ------------------------------------------------------------------
# MODULE 3: The OPPOSITE divergence -- correct steps, wrong synthesis
# ------------------------------------------------------------------

def task_check_synthesis_bug(case: dict) -> bool:
    correct_tool_result = lookup_flawed(case["reference"])  # tool itself is fine here
    final_answer = agent_with_synthesis_bug(case["reference"], correct_tool_result)
    expected_amount_str = f"{case['expected_amount']:,}"
    return expected_amount_str in final_answer

def step_check_synthesis_bug(case: dict) -> bool:
    # the STEP (tool call) is genuinely correct in this scenario --
    # the bug is ONLY in the synthesis step, which this check does not cover
    correct_tool_result = lookup_flawed(case["reference"])
    return correct_tool_result["found"] == case["expected_found"] and (
        correct_tool_result["amount"] == case["expected_amount"]
    )


task_accuracy_B = compute_task_level_accuracy(TEST_SET_STEP_HIGH_TASK_LOW, task_check_synthesis_bug)
step_accuracy_B = compute_step_level_accuracy(TEST_SET_STEP_HIGH_TASK_LOW, step_check_synthesis_bug)

print("=" * 70)
print("TEST SET B RESULTS -- SYNTHESIS BUG (tool call is CORRECT)")
print("=" * 70)

for case in TEST_SET_STEP_HIGH_TASK_LOW:
    tool_result = lookup_flawed(case["reference"])
    final_answer = agent_with_synthesis_bug(case["reference"], tool_result)
    print(f"\nReference: {case['reference']}")
    print(f"  Tool result (correct): {tool_result}")
    print(f"  Final answer (BUGGY synthesis): '{final_answer}'")

print(f"\nTask-level accuracy: {task_accuracy_B:.2f}  <- LOW (synthesis bug reports wrong amount)")
print(f"Step-level accuracy: {step_accuracy_B:.2f}  <- HIGH (the tool call itself was genuinely correct)")

print("\n" + "=" * 70)
print("BOTH DIVERGENCE PATTERNS, SIDE BY SIDE")
print("=" * 70)
print(f"Test Set A (flawed tool):     task={task_accuracy_A2:.2f}, step={step_accuracy_A2:.2f}  "
      f"(task > step -- 'right answer, wrong process')")
print(f"Test Set B (synthesis bug):   task={task_accuracy_B:.2f}, step={step_accuracy_B:.2f}  "
      f"(step > task -- correct steps, wrong final synthesis)")

print("\nCONFIRMED: both divergence directions this topic's theory describes")
print("are demonstrated with REAL, computed numbers from two genuinely")
print("different underlying bugs -- neither metric alone would have")
print("correctly diagnosed BOTH of these real, different failure modes.")

print("\nModule 3 complete. All Chapter 15 Topic 2 modules done.")
print()
print("=" * 70)
print("CHAPTER 15 TOPIC 2 -- KEY POINTS TO REMEMBER")
print("=" * 70)
print("""
  Task-level accuracy = does the FINAL OUTPUT match expectations.
  Step-level accuracy = does each INTERMEDIATE STEP'S result match
  expectations, independent of the final output.

  BOTH divergence directions demonstrated with REAL, computed numbers:
  task > step (flawed tool, correct-looking answers hiding a real bug)
  AND step > task (correct tool call, but a buggy synthesis step).

  NEITHER metric alone would have caught BOTH real bugs demonstrated
  here -- exactly why they must be reported TOGETHER, never one
  instead of the other.

  This generalizes Topic 1's single demonstration into a full
  framework, and sets up Topic 3's deeper focus on one specific,
  especially important step-level check: tool-selection correctness.
""")


TEST SET B RESULTS -- SYNTHESIS BUG (tool call is CORRECT)

Reference: BJ2019FD7717
  Tool result (correct): {'found': True, 'customer_name': 'Shobha Chopra', 'amount': 391000}
  Final answer (BUGGY synthesis): 'Your FD amount is Rs 999,999, registered to Shobha Chopra.'

Reference: BJ2022FD6918
  Tool result (correct): {'found': True, 'customer_name': 'Anita Mishra', 'amount': 160000}
  Final answer (BUGGY synthesis): 'Your FD amount is Rs 999,999, registered to Anita Mishra.'

Task-level accuracy: 0.00  <- LOW (synthesis bug reports wrong amount)
Step-level accuracy: 1.00  <- HIGH (the tool call itself was genuinely correct)

BOTH DIVERGENCE PATTERNS, SIDE BY SIDE
Test Set A (flawed tool):     task=1.00, step=0.67  (task > step -- 'right answer, wrong process')
Test Set B (synthesis bug):   task=0.00, step=1.00  (step > task -- correct steps, wrong final synthesis)

CONFIRMED: both divergence directions this topic's theory describes
are demonstrated with REAL, computed numbers from t